# Medical MNIST Classification - Training Notebook

This notebook runs the Medical MNIST classification pipeline.

**Dataset:** https://www.kaggle.com/datasets/andrewmvd/medical-mnist

**Classes:**
- AbdomenCT
- BreastMRI
- CXR (Chest X-Ray)
- ChestCT
- Hand
- HeadCT

## 1. Clone Repository

In [2]:
!git clone https://github.com/VinyVan/MedicalMNIST.git
%cd MedicalMNIST

fatal: destination path 'MedicalMNIST' already exists and is not an empty directory.
/content/MedicalMNIST


In [24]:
!git config --global user.email "kone22688@gmail.com"
!git config --global user.name "VinyVan"


In [28]:
# Note: Configure git remote URL without exposing credentials
# Use SSH or configure credentials via git config or environment variables

In [12]:
!git merge --abort
!git pull origin feature/initial-medical-mnist-pipeline --no-edit

fatal: There is no merge to abort (MERGE_HEAD missing).
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 508 bytes | 254.00 KiB/s, done.
From https://github.com/VinyVan/MedicalMNIST
 * branch            feature/initial-medical-mnist-pipeline -> FETCH_HEAD
   a7b87d8..052ad5c  feature/initial-medical-mnist-pipeline -> origin/feature/initial-medical-mnist-pipeline
Merge made by the 'ort' strategy.
 main.py            | 3 ++-
 src/data_loader.py | 3 ++-
 2 files changed, 4 insertions(+), 2 deletions(-)


## 2. Setup Environment

In [2]:
!pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 117.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 136.3 MB/s eta 0:00:0000:01


## 3. Configure Kaggle Credentials

Upload your kaggle.json file:

In [5]:
import os

# Set Kaggle credentials from environment variables
os.environ["KAGGLE_USERNAME"] = "vinymo"
os.environ["KAGGLE_KEY"] = "46444c4168069d3d6e2d1dc63f12884a"

print("Kaggle credentials configured!")
print(f"Username: {os.environ["KAGGLE_USERNAME"]}")

Kaggle credentials configured!
Username: vinymo


## 4. Download Medical MNIST Dataset

In [6]:
!python download_data.py

Medical MNIST Data Download

  /content/MedicalMNIST/data/raw

This may take a few minutes depending on your connection...
------------------------------------------------------------
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/medical-mnist
100% 84.8M/84.8M [00:05<00:00, 17.3MB/s]


Download complete!

Data verified successfully!
You can now run: python main.py --running_mode debug


## 5. Quick Data Exploration

In [8]:
import sys
sys.path.append('src')
from config.paths import Paths
from src.data_loader import load_medical_mnist_data, check_data_exists

if check_data_exists(Paths.RAW_DATA_DIR):
    image_paths, labels, class_names = load_medical_mnist_data(Paths.RAW_DATA_DIR)
    print(f'Total images: {len(image_paths)}')
    print(f'Classes: {class_names}')
    from collections import Counter
    for cls_idx, count in sorted(Counter(labels).items()):
        print(f'  {class_names[cls_idx]}: {count}')
else:
    print('Data not found. Please check download step.')

Total images: 58954
Classes: ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
  AbdomenCT: 10000
  BreastMRI: 8954
  CXR: 10000
  ChestCT: 10000
  Hand: 10000
  HeadCT: 10000


In [35]:
!cat main.py

"""
Main entry point for Medical MNIST Classification Pipeline
Run via: python main.py --running_mode debug --model cnn

Dataset: Medical MNIST (6 classes of medical images)
- AbdomenCT
- BreastMRI
- CXR (Chest X-Ray)
- ChestCT
- Hand
- HeadCT

Image size: 64x64 pixels, Grayscale
"""

import argparse
import sys
from pathlib import Path
from datetime import datetime

# Add src to path
sys.path.append(str(Path(__file__).parent / 'src'))

import torch
import numpy as np
import pandas as pd

from config.config import Config
from config.paths import Paths
from src.data_loader import (
    load_medical_mnist_data,
    create_data_loaders,
    create_cv_splits,
    download_medical_mnist_kaggle,
    check_data_exists,
    split_train_test
)
from src.modeling.models import get_model, print_model_summary, count_parameters
from src.training.trainer import CVTrainer
from src.utils import set_seed, setup_logging, get_device
from sklearn.metrics import accuracy_score, classification_report, confusi

## 6. Training - Debug Mode (Quick Test)

In [14]:
!python main.py --running_mode train --model cnn --fold 3 --epochs 15

MEDICAL MNIST CLASSIFICATION PIPELINE
Mode: train
Model: cnn
Device: cuda
Folds: 3
CV Strategy: stratified_kfold
Epochs: 30
Batch Size: 64
Learning Rate: 0.001
2026-05-11 07:14:18,589 - src.utils - INFO - Starting pipeline: mode=train, model=cnn
CUDA available: True
CUDA device: Tesla T4

1. LOADING DATA
----------------------------------------
2026-05-11 07:14:18,614 - src.data_loader - INFO - Loading Medical MNIST data from /content/MedicalMNIST/data/raw
2026-05-11 07:14:18,614 - src.data_loader - INFO - Found classes: ['AbdomenCT', 'BreastMRI', 'CXR', 'ChestCT', 'Hand', 'HeadCT']
2026-05-11 07:14:18,800 - src.data_loader - INFO -   AbdomenCT: 10000 images
2026-05-11 07:14:18,841 - src.data_loader - INFO -   BreastMRI: 8954 images
2026-05-11 07:14:18,884 - src.data_loader - INFO -   CXR: 10000 images
2026-05-11 07:14:18,926 - src.data_loader - INFO -   ChestCT: 10000 images
2026-05-11 07:14:19,116 - src.data_loader - INFO -   Hand: 10000 images
2026-05-11 07:14:19,165 - src.data_load

## 7. Full Training

In [ ]:
!python main.py --running_mode train --model cnn --fold 5 --epochs 30

In [ ]:
!python main.py --running_mode train --model resnet18 --fold 5 --epochs 30

In [ ]:
!python main.py --running_mode train --model efficientnet_b0 --fold 5 --epochs 30

## 8. Check Results

In [15]:
!ls -la outputs/models/
!ls -la outputs/oof/
!ls -la outputs/logs/

total 152672
drwxr-xr-x 2 root root     4096 May 11 06:18 .
drwxr-xr-x 6 root root     4096 May 11 06:17 ..
-rw-r--r-- 1 root root 52107019 May 11 07:38 model_fold_0.pth
-rw-r--r-- 1 root root 52107019 May 11 07:53 model_fold_1.pth
-rw-r--r-- 1 root root 52107019 May 11 08:08 model_fold_2.pth
total 2512
drwxr-xr-x 2 root root    4096 May 11 08:08 .
drwxr-xr-x 6 root root    4096 May 11 06:17 ..
-rw-r--r-- 1 root root 2560328 May 11 08:08 oof_predictions.csv
total 64
drwxr-xr-x 2 root root  4096 May 11 07:14 .
drwxr-xr-x 6 root root  4096 May 11 06:17 ..
-rw-r--r-- 1 root root  5472 May 11 06:18 run_debug_cnn_20260511_061734.log
-rw-r--r-- 1 root root  5567 May 11 06:45 run_debug_cnn_20260511_064502.log
-rw-r--r-- 1 root root  5567 May 11 06:56 run_debug_cnn_20260511_065622.log
-rw-r--r-- 1 root root  5567 May 11 07:04 run_debug_cnn_20260511_070435.log
-rw-r--r-- 1 root root  5711 May 11 07:12 run_debug_cnn_20260511_071224.log
-rw-r--r-- 1 root root 13973 May 11 08:08 run_train_cnn_2026

In [18]:
!git add .
!git commit -m "Simple CNN trained"
!git push origin feature/initial-medical-mnist-pipeline

On branch feature/initial-medical-mnist-pipeline
nothing to commit, working tree clean
fatal: could not read Username for 'https://github.com': No such device or address


In [29]:
!git push origin feature/initial-medical-mnist-pipeline

Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (14/14), 137.28 MiB | 9.32 MiB/s, done.
Total 14 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/VinyVan/MedicalMNIST.git
   052ad5c..3b56cb9  feature/initial-medical-mnist-pipeline -> feature/initial-medical-mnist-pipeline
